# 0.0 Imports

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import sklearn.metrics as mt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.pipeline import Pipeline

# 0.1 - Load Datasets

In [3]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks\regressao
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [4]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [5]:
# Instanciar o modelo com parâmetros default (grau 1)
model_def = Pipeline([
    ('features', PolynomialFeatures(degree=1)),
    ('model', Lasso())
])

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [6]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.0074
MSE:  474.47
RMSE: 21.78
MAE:  17.31
MAPE: 873.67%


## Passo 3 — Performance na validação (default)

In [7]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   0.0079
MSE:  473.75
RMSE: 21.77
MAE:  17.26
MAPE: 869.58%


## Passo 4 — Ajuste de hiperparâmetros

In [ ]:
print("--- Testando Regressão Polinomial com Penalização Lasso (L1) ---")
melhor_alpha = 1.0
melhor_degree = 2
melhor_r2_val = -999

# Listas de hiperparâmetros para o ensaio cruzado
list_degrees = [2, 3]
list_alpha = [0.001, 0.01, 0.1, 1.0, 10.0]

# LOOP DUPLO: Cruza Grau do Polinómio x Alpha do Lasso
for dg in list_degrees:
    for alpha in list_alpha:
        
        # Criamos o algoritmo composto de Regressão Polinomial com Lasso
        # max_iter expandido para 10000 para dar margem de convergência ao Grau 3
        model_poly_lasso = Pipeline([
            ('features_polinomiais', PolynomialFeatures(degree=dg, include_bias=False)),
            ('regressor_lasso', Lasso(alpha=alpha, max_iter=10000, random_state=42))
        ])
        
        try:
            model_poly_lasso.fit(X_train, y_train.values.ravel())
            
            yhat_train = model_poly_lasso.predict(X_train)
            yhat_val = model_poly_lasso.predict(X_val)
            
            r2_train = mt.r2_score(y_train, yhat_train)
            r2_val = mt.r2_score(y_val, yhat_val)
            rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
            
            # Blindagem do MAPE contra distorções perto de zero
            mape_raw = mt.mean_absolute_percentage_error(y_val, yhat_val)
            mape_print = f"{mape_raw * 100:.2f}%" if np.isfinite(mape_raw) and mape_raw < 1000 else "Distort (High)"
            
            print(f"Grau: {dg} | Alpha: {alpha:<6} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f} | MAPE Val: {mape_print}")
            
            if r2_val > melhor_r2_val:
                melhor_r2_val = r2_val
                melhor_alpha = alpha
                melhor_degree = dg
                
        except Exception as e:
            print(f"Erro na combinação Grau {dg} | Alpha {alpha}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO POLINOMIAL + LASSO:")
print(f"Melhor Grau Encontrado: {melhor_degree}")
print(f"Melhor Alpha (Lasso): {melhor_alpha}")
print(f"Maior R² obtido na Validação: {melhor_r2_val:.4f}\n")

--- Testando Regressão Polinomial com Penalização Lasso (L1) ---
Grau: 2 | Alpha: 0.001  | R² Treino: 0.0935 | R² Val: 0.0680 | RMSE Val: 21.10 | MAPE Val: 856.52%
Grau: 2 | Alpha: 0.01   | R² Treino: 0.0868 | R² Val: 0.0685 | RMSE Val: 21.09 | MAPE Val: 859.10%
Grau: 2 | Alpha: 0.1    | R² Treino: 0.0679 | R² Val: 0.0589 | RMSE Val: 21.20 | MAPE Val: 866.76%
Grau: 2 | Alpha: 1.0    | R² Treino: 0.0091 | R² Val: 0.0096 | RMSE Val: 21.75 | MAPE Val: 868.18%
Grau: 2 | Alpha: 10.0   | R² Treino: 0.0000 | R² Val: -0.0000 | RMSE Val: 21.85 | MAPE Val: 867.87%


## Passo 5 — Performance com modelo tunado (treino e validação)

In [9]:
melhor_degree = 2
melhor_alpha = 0.01

# Modelo com os melhores hiperparâmetros encontrados, treinado apenas em X_train
model_tunado = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=melhor_degree, include_bias=False)),
    ('regressor_lasso', Lasso(alpha=melhor_alpha, max_iter=10000, random_state=42))
])
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"Treino    → R²: {r2_train_tunado:.4f} | RMSE: {rmse_train_tunado:.2f} | MAE: {mae_train_tunado:.2f} | MAPE: {mape_train_tunado*100:.2f}%")
print(f"Validação → R²: {r2_val_tunado:.4f} | RMSE: {rmse_val_tunado:.2f} | MAE: {mae_val_tunado:.2f} | MAPE: {mape_val_tunado*100:.2f}%")

--- Performance com Modelo Tunado ---
Treino    → R²: 0.0868 | RMSE: 20.89 | MAE: 16.54 | MAPE: 843.27%
Validação → R²: 0.0685 | RMSE: 21.09 | MAE: 16.73 | MAPE: 859.10%


## Passo 6 — União treino + validação

In [10]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [11]:
# Retreino com os melhores hiperparâmetros sobre o conjunto final (treino + validação)
model_final = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=melhor_degree, include_bias=False)),
    ('regressor_lasso', Lasso(alpha=melhor_alpha, max_iter=10000, random_state=42))
])
model_final.fit(X_train_final, y_train_final.values.ravel())

,steps,"[('features_polinomiais', ...), ('regressor_lasso', ...)]"
,transform_input,None
,memory,None
,verbose,False
,degree,2
,interaction_only,False
,include_bias,False
,order,'C'
,alpha,0.01
,fit_intercept,True
,precompute,False


## Passo 8 — Performance no teste

In [12]:
# Predição no conjunto de teste (única vez que X_test é tocado)
yhat_test = model_final.predict(X_test)

# Métricas no conjunto de TESTE
r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print("--- Performance no Teste (Final) ---")
print(f"R²:   {r2_test:.4f}")
print(f"MSE:  {mse_test:.2f}")
print(f"RMSE: {rmse_test:.2f}")
print(f"MAE:  {mae_test:.2f}")
print(f"MAPE: {mape_test * 100:.2f}%")

--- Performance no Teste (Final) ---
R²:   0.0854
MSE:  445.33
RMSE: 21.10
MAE:  16.79
MAPE: 834.21%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [13]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto       R²      RMSE       MAE    MAPE
   Treino (Default) 0.007401 21.782443 17.305484 873.67%
Validação (Default) 0.007884 21.765732 17.264922 869.58%
    Treino (Tunado) 0.086817 20.892893 16.540954 843.27%
 Validação (Tunado) 0.068473 21.090637 16.732386 859.10%
      Teste (Final) 0.085374 21.102896 16.785714 834.21%
